# 07 - Results Interpretation

**Purpose:** consolidate the final results around the project question:

> Can time-series history, social-network exposure, review-language signals, and stronger scikit-learn model families help forecast short-term shifts in community attention toward local Yelp businesses?

This interpretation uses the corrected forecasting cohort: **876 businesses** and **68,249 business-month rows** from the 2015-2021 modeling window, excluding rows before each business's first observed modeling-window review.

In [1]:
from pathlib import Path
import json

import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "new_orleans"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

METRICS_OUTPUT_PATH = OUTPUTS_DIR / "forecasting_metrics.csv"
PREDICTIONS_OUTPUT_PATH = OUTPUTS_DIR / "forecasting_predictions.csv"
PULSE_METRICS_OUTPUT_PATH = OUTPUTS_DIR / "attention_pulse_metrics.csv"
PULSE_PREDICTIONS_OUTPUT_PATH = OUTPUTS_DIR / "attention_pulse_predictions.csv"
PULSE_TOPK_OUTPUT_PATH = OUTPUTS_DIR / "attention_pulse_topk_metrics.csv"
PULSE_CALIBRATION_OUTPUT_PATH = OUTPUTS_DIR / "attention_pulse_calibration.csv"
GRAPH_SUMMARY_PATH = PROCESSED_DIR / "social_graph_summary.json"
FEATURE_SUMMARY_PATH = PROCESSED_DIR / "forecasting_feature_summary.json"
FORECASTING_DATASET_PATH = PROCESSED_DIR / "forecasting_dataset.csv"
COHORT_BUSINESSES_PATH = PROCESSED_DIR / "forecasting_cohort_businesses.csv"
PULSE_PREDECESSOR_OUTPUT_PATH = OUTPUTS_DIR / "pulse_predecessor_analysis.csv"
CASE_STUDIES_OUTPUT_PATH = OUTPUTS_DIR / "attention_pulse_case_studies.csv"

metrics = pd.read_csv(METRICS_OUTPUT_PATH)
predictions = pd.read_csv(PREDICTIONS_OUTPUT_PATH)
pulse_metrics = pd.read_csv(PULSE_METRICS_OUTPUT_PATH)
pulse_predictions = pd.read_csv(PULSE_PREDICTIONS_OUTPUT_PATH)
pulse_topk_metrics = pd.read_csv(PULSE_TOPK_OUTPUT_PATH)
pulse_calibration = pd.read_csv(PULSE_CALIBRATION_OUTPUT_PATH)
modeling = pd.read_csv(FORECASTING_DATASET_PATH)
business_lookup = pd.read_csv(COHORT_BUSINESSES_PATH)[["business_id", "name", "categories"]]
with GRAPH_SUMMARY_PATH.open("r", encoding="utf-8") as file:
    graph_summary = json.load(file)
with FEATURE_SUMMARY_PATH.open("r", encoding="utf-8") as file:
    feature_summary = json.load(file)

metrics.sort_values(["split", "WAPE", "MAE"])

,split,task,model,train_period,validation_period,test_period,rows,MAE,RMSE,WAPE,model_family,feature_count
0,primary_covid_test,review_count_regression,Baseline: last month,2015-02 to 2018-12,2019-01 to 2019-12,2020-01 to 2021-12,21024,1.601170,3.007150,0.664292,temporal_baseline,0
1,primary_covid_test,review_count_regression,Baseline: rolling 3-month avg,2015-02 to 2018-12,2019-01 to 2019-12,2020-01 to 2021-12,21024,1.646071,3.364436,0.682921,temporal_baseline,0
2,primary_covid_test,review_count_regression,ML: historical + SNA,2015-02 to 2018-12,2019-01 to 2019-12,2020-01 to 2021-12,21024,1.970705,3.525411,0.817604,RandomForestRegressor,33
3,primary_covid_test,review_count_regression,ML: historical + NLP,2015-02 to 2018-12,2019-01 to 2019-12,2020-01 to 2021-12,21024,1.978451,3.503941,0.820818,RandomForestRegressor,68
4,primary_covid_test,review_count_regression,ML: historical + business,2015-02 to 2018-12,2019-01 to 2019-12,2020-01 to 2021-12,21024,1.997847,3.636709,0.828865,RandomForestRegressor,16
5,primary_covid_test,review_count_regression,ML: historical,2015-02 to 2018-12,2019-01 to 2019-12,2020-01 to 2021-12,21024,2.004549,3.607005,0.831646,RandomForestRegressor,10
6,primary_covid_test,review_count_regression,ML: all modalities,2015-02 to 2018-12,2019-01 to 2019-12,2020-01 to 2021-12,21024,2.053257,3.599565,0.851854,RandomForestRegressor,97
7,primary_covid_test,review_count_regression,ML: Poisson all modalities,2015-02 to 2018-12,2019-01 to 2019-12,2020-01 to 2021-12,21024,2.099909,4.102147,0.871208,PoissonRegressor,97
8,primary_covid_test,review_count_regression,ML: HGB selected top 20,2015-02 to 2018-12,2019-01 to 2019-12,2020-01 to 2021-12,21024,2.180942,3.520737,0.904827,SelectKBest + HistGradientBoostingRegressor,97
9,primary_covid_test,review_count_regression,ML: HGB all modalities,2015-02 to 2018-12,2019-01 to 2019-12,2020-01 to 2021-12,21024,2.254200,4.005916,0.935220,HistGradientBoostingRegressor,97


## Regression Interpretation

Summarize which model family performs best in each time split and whether stronger alternatives improve on the simple temporal baselines. This pass compares Random Forests, HistGradientBoosting, Poisson count models, and selected-feature variants.

In [2]:
regression_summary_rows = []
for split_name, split_metrics in metrics.groupby("split"):
    ranked = split_metrics.sort_values("WAPE").reset_index(drop=True)
    best = ranked.iloc[0]
    hist = split_metrics[split_metrics["model"] == "ML: historical"].iloc[0]
    business = split_metrics[split_metrics["model"] == "ML: historical + business"].iloc[0]
    sna = split_metrics[split_metrics["model"] == "ML: historical + SNA"].iloc[0]
    nlp = split_metrics[split_metrics["model"] == "ML: historical + NLP"].iloc[0]
    all_modalities = split_metrics[split_metrics["model"] == "ML: all modalities"].iloc[0]
    hgb_all = split_metrics[split_metrics["model"] == "ML: HGB all modalities"].iloc[0]
    poisson_all = split_metrics[split_metrics["model"] == "ML: Poisson all modalities"].iloc[0]
    selected_hgb = split_metrics[split_metrics["model"] == "ML: HGB selected top 20"].iloc[0]
    regression_summary_rows.append({
        "split": split_name,
        "best_model": best["model"],
        "best_family": best["model_family"],
        "best_WAPE": best["WAPE"],
        "historical_WAPE": hist["WAPE"],
        "business_WAPE": business["WAPE"],
        "sna_WAPE": sna["WAPE"],
        "nlp_WAPE": nlp["WAPE"],
        "all_modalities_WAPE": all_modalities["WAPE"],
        "hgb_all_WAPE": hgb_all["WAPE"],
        "poisson_all_WAPE": poisson_all["WAPE"],
        "selected_hgb_WAPE": selected_hgb["WAPE"],
        "best_vs_last_month_relative_change": (best["WAPE"] - split_metrics[split_metrics["model"] == "Baseline: last month"].iloc[0]["WAPE"]) / split_metrics[split_metrics["model"] == "Baseline: last month"].iloc[0]["WAPE"],
        "all_vs_historical_relative_change": (all_modalities["WAPE"] - hist["WAPE"]) / hist["WAPE"],
        "all_vs_business_relative_change": (all_modalities["WAPE"] - business["WAPE"]) / business["WAPE"],
    })
regression_summary = pd.DataFrame(regression_summary_rows)
regression_summary

,split,best_model,best_family,best_WAPE,historical_WAPE,business_WAPE,sna_WAPE,nlp_WAPE,all_modalities_WAPE,hgb_all_WAPE,poisson_all_WAPE,selected_hgb_WAPE,best_vs_last_month_relative_change,all_vs_historical_relative_change,all_vs_business_relative_change
0,primary_covid_test,Baseline: last month,temporal_baseline,0.664292,0.831646,0.828865,0.817604,0.820818,0.851854,0.935220,0.871208,0.904827,0.000000,0.024299,0.027735
1,secondary_pre_covid_test,ML: HGB all modalities,HistGradientBoostingRegressor,0.380617,0.388882,0.381904,0.388126,0.388565,0.385004,0.380617,0.449454,0.392028,-0.149063,-0.009972,0.008115


## Pulse Interpretation

Summarize pulse-classification performance with class balance and probability quality in mind:

- F1 uses a cutoff tuned on the validation period;
- PR-AUC and top-k metrics assess ranking quality for rare pulse events;
- Brier score and calibration bins assess whether pulse probabilities behave like useful risk estimates.

In [3]:
# Pulse summaries emphasize rare-event detection and probability quality rather than overall accuracy.
pulse_summary_rows = []
for split_name, split_metrics in pulse_metrics.groupby("split"):
    ranked = split_metrics.sort_values(["F1", "PR_AUC"], ascending=[False, False]).reset_index(drop=True)
    best = ranked.iloc[0]
    hist = split_metrics[split_metrics["model"] == "ML: historical"].iloc[0]
    business = split_metrics[split_metrics["model"] == "ML: historical + business"].iloc[0]
    sna = split_metrics[split_metrics["model"] == "ML: historical + SNA"].iloc[0]
    nlp = split_metrics[split_metrics["model"] == "ML: historical + NLP"].iloc[0]
    all_modalities = split_metrics[split_metrics["model"] == "ML: all modalities"].iloc[0]
    hgb_all = split_metrics[split_metrics["model"] == "ML: HGB all modalities"].iloc[0]
    logistic_all = split_metrics[split_metrics["model"] == "ML: Logistic all modalities"].iloc[0]
    selected_hgb = split_metrics[split_metrics["model"] == "ML: HGB selected top 20"].iloc[0]
    best_brier = split_metrics.sort_values("Brier").iloc[0]

    split_topk_10 = pulse_topk_metrics[
        (pulse_topk_metrics["split"] == split_name)
        & (pulse_topk_metrics["k_fraction"] == 0.10)
    ].copy()
    best_topk_10 = split_topk_10.sort_values(["precision_at_k", "recall_at_k"], ascending=[False, False]).iloc[0]
    all_topk_10 = split_topk_10[split_topk_10["model"] == "ML: all modalities"].iloc[0]

    pulse_summary_rows.append({
        "split": split_name,
        "positive_rate": all_modalities["positive_rate"],
        "best_model": best["model"],
        "best_family": best["model_family"],
        "best_F1": best["F1"],
        "best_PR_AUC": best["PR_AUC"],
        "best_Brier": best["Brier"],
        "best_threshold": best["decision_threshold"],
        "best_validation_F1": best["validation_F1"],
        "best_brier_model": best_brier["model"],
        "best_brier": best_brier["Brier"],
        "historical_F1": hist["F1"],
        "business_F1": business["F1"],
        "sna_F1": sna["F1"],
        "nlp_F1": nlp["F1"],
        "all_modalities_F1": all_modalities["F1"],
        "all_modalities_PR_AUC": all_modalities["PR_AUC"],
        "hgb_all_F1": hgb_all["F1"],
        "hgb_all_PR_AUC": hgb_all["PR_AUC"],
        "logistic_all_F1": logistic_all["F1"],
        "logistic_all_PR_AUC": logistic_all["PR_AUC"],
        "selected_hgb_F1": selected_hgb["F1"],
        "selected_hgb_PR_AUC": selected_hgb["PR_AUC"],
        "selected_hgb_Brier": selected_hgb["Brier"],
        "all_modalities_threshold": all_modalities["decision_threshold"],
        "best_precision_at_10pct_model": best_topk_10["model"],
        "best_precision_at_10pct": best_topk_10["precision_at_k"],
        "best_recall_at_10pct": best_topk_10["recall_at_k"],
        "all_modalities_precision_at_10pct": all_topk_10["precision_at_k"],
        "all_modalities_recall_at_10pct": all_topk_10["recall_at_k"],
    })
pulse_summary = pd.DataFrame(pulse_summary_rows)
pulse_summary

,split,positive_rate,best_model,best_family,best_F1,best_PR_AUC,best_Brier,best_threshold,best_validation_F1,best_brier_model,...,logistic_all_PR_AUC,selected_hgb_F1,selected_hgb_PR_AUC,selected_hgb_Brier,all_modalities_threshold,best_precision_at_10pct_model,best_precision_at_10pct,best_recall_at_10pct,all_modalities_precision_at_10pct,all_modalities_recall_at_10pct
0,primary_covid_test,0.102407,Baseline: rising recent activity,rule_baseline,0.280882,0.148129,0.200200,0.500,0.215558,ML: HGB selected top 20,...,0.256856,0.275767,0.230490,0.092019,0.405,ML: Logistic all modalities,0.311935,0.304691,0.198288,0.193683
1,secondary_pre_covid_test,0.131957,ML: HGB selected top 20,SelectKBest + HistGradientBoostingClassifier,0.308690,0.237247,0.110175,0.155,0.382133,ML: HGB all modalities,...,0.210589,0.308690,0.237247,0.110175,0.425,ML: historical,0.301331,0.228551,0.289924,0.219899


## Final Model Comparison Summary

This table is the report-ready bridge between metrics and interpretation. It combines the best review-count model, the best attention-pulse model, simple baseline comparisons, and a compact modality takeaway for each chronological split.


In [4]:
comparison_rows = []
for split_name in regression_summary["split"]:
    split_regression = metrics[metrics["split"] == split_name].copy()
    split_pulse = pulse_metrics[pulse_metrics["split"] == split_name].copy()
    reg = regression_summary[regression_summary["split"] == split_name].iloc[0]
    pulse = pulse_summary[pulse_summary["split"] == split_name].iloc[0]

    last_month = split_regression[split_regression["model"] == "Baseline: last month"].iloc[0]
    best_pulse_baseline = (
        split_pulse[split_pulse["model_family"] == "rule_baseline"]
        .sort_values(["F1", "PR_AUC"], ascending=[False, False])
        .iloc[0]
    )

    sna_wape_change = (reg["sna_WAPE"] - reg["historical_WAPE"]) / reg["historical_WAPE"]
    nlp_wape_change = (reg["nlp_WAPE"] - reg["historical_WAPE"]) / reg["historical_WAPE"]
    pulse_sna_f1_change = pulse["sna_F1"] - pulse["historical_F1"]
    pulse_nlp_f1_change = pulse["nlp_F1"] - pulse["historical_F1"]

    if split_name == "primary_covid_test":
        modality_takeaway = "COVID disruption makes raw count forecasting baseline-heavy; multimodal features help more as pulse ranking/interpretation than as count lift."
    else:
        modality_takeaway = "Pre-COVID patterns are more learnable; business context helps counts, while selected models are cleaner than using every modality blindly."

    comparison_rows.append({
        "split": split_name,
        "best_review_count_model": reg["best_model"],
        "best_review_count_WAPE": reg["best_WAPE"],
        "last_month_baseline_WAPE": last_month["WAPE"],
        "best_count_vs_last_month_pct": reg["best_vs_last_month_relative_change"] * 100,
        "best_pulse_model": pulse["best_model"],
        "best_pulse_F1": pulse["best_F1"],
        "best_pulse_PR_AUC": pulse["best_PR_AUC"],
        "best_pulse_baseline": best_pulse_baseline["model"],
        "best_pulse_baseline_F1": best_pulse_baseline["F1"],
        "best_pulse_vs_baseline_F1_delta": pulse["best_F1"] - best_pulse_baseline["F1"],
        "best_top10_precision_model": pulse["best_precision_at_10pct_model"],
        "best_top10_precision": pulse["best_precision_at_10pct"],
        "sna_vs_historical_WAPE_pct": sna_wape_change * 100,
        "nlp_vs_historical_WAPE_pct": nlp_wape_change * 100,
        "pulse_sna_vs_historical_F1_delta": pulse_sna_f1_change,
        "pulse_nlp_vs_historical_F1_delta": pulse_nlp_f1_change,
        "modality_takeaway": modality_takeaway,
    })

model_comparison_summary = pd.DataFrame(comparison_rows)
MODEL_COMPARISON_SUMMARY_PATH = OUTPUTS_DIR / "model_comparison_summary.csv"
model_comparison_summary.to_csv(MODEL_COMPARISON_SUMMARY_PATH, index=False)
model_comparison_summary


,split,best_review_count_model,best_review_count_WAPE,last_month_baseline_WAPE,best_count_vs_last_month_pct,best_pulse_model,best_pulse_F1,best_pulse_PR_AUC,best_pulse_baseline,best_pulse_baseline_F1,best_pulse_vs_baseline_F1_delta,best_top10_precision_model,best_top10_precision,sna_vs_historical_WAPE_pct,nlp_vs_historical_WAPE_pct,pulse_sna_vs_historical_F1_delta,pulse_nlp_vs_historical_F1_delta,modality_takeaway
0,primary_covid_test,Baseline: last month,0.664292,0.664292,0.000000,Baseline: rising recent activity,0.280882,0.148129,Baseline: rising recent activity,0.280882,0.000000,ML: Logistic all modalities,0.311935,-1.688393,-1.301941,0.002225,-0.027662,COVID disruption makes raw count forecasting b...
1,secondary_pre_covid_test,ML: HGB all modalities,0.380617,0.447291,-14.906337,ML: HGB selected top 20,0.308690,0.237247,Baseline: rising recent activity,0.215558,0.093132,ML: historical,0.301331,-0.194192,-0.081302,0.002314,-0.009962,Pre-COVID patterns are more learnable; busines...


## What Precedes An Attention Pulse?

This comparison checks whether pulse rows have different recent conditions than non-pulse rows before the target month arrives. It focuses on interpretable signals: review momentum, recent text volume/language, reviewer centrality, and social exposure.


In [5]:
pre_pulse_features = [
    "prev_month_reviews",
    "rolling_3_avg",
    "rolling_6_avg",
    "same_month_previous_year",
    "recent_text_review_count",
    "avg_recent_review_word_count",
    "share_recent_positive_language",
    "share_recent_negative_language",
    "recent_text_avg_stars",
    "recent_reviewer_count",
    "avg_reviewer_weighted_degree",
    "max_reviewer_weighted_pagerank",
    "fraction_repeat_reviewers",
    "reviewer_community_diversity",
    "fraction_active_to_date_reviewers",
    "avg_reviewer_recency_months",
] + sorted([column for column in modeling.columns if column.startswith("tfidf_recent_")])[:20]

available_pre_pulse_features = [feature for feature in pre_pulse_features if feature in modeling.columns]
pulse_predecessor_summary = (
    modeling.groupby("attention_pulse")[available_pre_pulse_features]
    .mean()
    .T
    .rename(columns={0: "non_pulse_mean", 1: "pulse_mean"})
)
pulse_predecessor_summary["absolute_difference"] = pulse_predecessor_summary["pulse_mean"] - pulse_predecessor_summary["non_pulse_mean"]
pulse_predecessor_summary["relative_lift_vs_non_pulse"] = pulse_predecessor_summary["pulse_mean"] / pulse_predecessor_summary["non_pulse_mean"].replace(0, np.nan)
pulse_predecessor_summary = pulse_predecessor_summary.sort_values("absolute_difference", ascending=False)
pulse_predecessor_summary_output = pulse_predecessor_summary.reset_index().rename(columns={"index": "feature"})
pulse_predecessor_summary_output.to_csv(PULSE_PREDECESSOR_OUTPUT_PATH, index=False)
pulse_predecessor_summary_output.head(20)


attention_pulse,feature,non_pulse_mean,pulse_mean,absolute_difference,relative_lift_vs_non_pulse
0,avg_recent_review_word_count,87.215370,94.201212,6.985842,1.080099
1,avg_reviewer_weighted_degree,31.289212,34.597184,3.307972,1.105722
2,recent_text_avg_stars,3.655975,3.937210,0.281235,1.076925
3,prev_month_reviews,5.010349,5.145895,0.135546,1.027053
4,reviewer_community_diversity,2.118906,2.181838,0.062932,1.029700
5,share_recent_positive_language,0.721067,0.779101,0.058034,1.080483
6,tfidf_recent_amazing,0.227545,0.254093,0.026548,1.116670
7,tfidf_recent_best,0.224018,0.250238,0.026219,1.117041
8,tfidf_recent_got,0.218519,0.244084,0.025565,1.116992
9,tfidf_recent_definitely,0.212040,0.237255,0.025214,1.118914


## Example Pulse Case Studies

The table below selects a few high-signal business-months from the primary COVID-era test split. It includes correct and incorrect pulse predictions so the report can discuss what the model sees, where it succeeds, and where community attention remains hard to anticipate.


In [6]:
case_model = "ML: HGB selected top 20"
case_split = "primary_covid_test"
case_pool = pulse_predictions[
    (pulse_predictions["split"] == case_split)
    & (pulse_predictions["model"] == case_model)
].copy()
if case_pool.empty:
    case_model = "ML: all modalities"
    case_pool = pulse_predictions[
        (pulse_predictions["split"] == case_split)
        & (pulse_predictions["model"] == case_model)
    ].copy()

case_pool["case_type"] = np.select(
    [
        case_pool["attention_pulse"].eq(1) & case_pool["prediction"].eq(1),
        case_pool["attention_pulse"].eq(1) & case_pool["prediction"].eq(0),
        case_pool["attention_pulse"].eq(0) & case_pool["prediction"].eq(1),
        case_pool["attention_pulse"].eq(0) & case_pool["prediction"].eq(0),
    ],
    ["true_positive_pulse", "missed_pulse", "false_alarm", "true_negative"],
    default="other",
)

case_specs = [
    ("true_positive_pulse", False),
    ("missed_pulse", False),
    ("false_alarm", False),
    ("true_positive_pulse", True),
    ("missed_pulse", True),
]
case_frames = []
used_index = set()
for case_type, ascending_probability in case_specs:
    candidates = case_pool[case_pool["case_type"] == case_type].copy()
    if candidates.empty:
        continue
    candidates = candidates.sort_values(
        ["probability", "target_next_month_reviews"],
        ascending=[ascending_probability, False],
    )
    for idx, row in candidates.iterrows():
        if idx not in used_index:
            used_index.add(idx)
            case_frames.append(row.to_frame().T)
            break

case_studies = pd.concat(case_frames, ignore_index=True) if case_frames else case_pool.head(5)
case_studies = (
    case_studies
    .merge(business_lookup, on="business_id", how="left")
    .merge(
        modeling[[
            "business_id",
            "feature_month_str",
            "rolling_3_avg",
            "rolling_6_avg",
            "recent_text_review_count",
            "share_recent_positive_language",
            "share_recent_negative_language",
            "recent_reviewer_count",
            "avg_reviewer_weighted_degree",
            "max_reviewer_weighted_pagerank",
            "fraction_repeat_reviewers",
            "reviewer_community_diversity",
        ]],
        on=["business_id", "feature_month_str"],
        how="left",
    )
)

case_studies["interpretation_note"] = np.select(
    [
        case_studies["case_type"].eq("true_positive_pulse"),
        case_studies["case_type"].eq("missed_pulse"),
        case_studies["case_type"].eq("false_alarm"),
    ],
    [
        "Model correctly flagged a future pulse; inspect recent momentum, reviewers, and language as plausible warning signals.",
        "A pulse occurred despite a low model decision; useful for discussing hidden offline events or weak digital precursors.",
        "Model expected a pulse that did not materialize; useful for discussing noisy attention signals and threshold trade-offs.",
    ],
    default="Stable non-pulse example.",
)

case_columns = [
    "case_type",
    "model",
    "business_id",
    "name",
    "target_month_str",
    "attention_pulse",
    "prediction",
    "probability",
    "decision_threshold",
    "target_next_month_reviews",
    "pulse_baseline_reviews",
    "pulse_relative_lift",
    "rolling_3_avg",
    "rolling_6_avg",
    "recent_text_review_count",
    "share_recent_positive_language",
    "share_recent_negative_language",
    "recent_reviewer_count",
    "avg_reviewer_weighted_degree",
    "max_reviewer_weighted_pagerank",
    "fraction_repeat_reviewers",
    "reviewer_community_diversity",
    "interpretation_note",
]
case_studies = case_studies[case_columns]
case_studies.to_csv(CASE_STUDIES_OUTPUT_PATH, index=False)
case_studies


,case_type,model,business_id,name,target_month_str,attention_pulse,prediction,probability,decision_threshold,target_next_month_reviews,...,rolling_6_avg,recent_text_review_count,share_recent_positive_language,share_recent_negative_language,recent_reviewer_count,avg_reviewer_weighted_degree,max_reviewer_weighted_pagerank,fraction_repeat_reviewers,reviewer_community_diversity,interpretation_note
0,true_positive_pulse,ML: HGB selected top 20,DcBLYSvOuWcNReolRVr12A,Drago's Seafood Restaurant,2021-03,1,1,0.692259,0.2,9.0,...,2.000000,7.0,0.857143,0.142857,7.0,2.205979,0.000037,0.714286,1.0,Model correctly flagged a future pulse; inspec...
1,missed_pulse,ML: HGB selected top 20,x4K6aMaOYvGhC5jhFJP2Ag,Pho Tau Bay Restaurant,2020-12,1,0,0.199988,0.2,5.0,...,0.500000,3.0,1.000000,0.333333,3.0,14.064971,0.000086,0.666667,1.0,A pulse occurred despite a low model decision;...
2,false_alarm,ML: HGB selected top 20,XnQ84ylyAZwh-XfHGGNBbQ,Coop's Place,2021-06,0,1,0.694569,0.2,3.0,...,2.666667,15.0,0.733333,0.266667,15.0,0.356346,0.000024,0.466667,2.0,Model expected a pulse that did not materializ...
3,true_positive_pulse,ML: HGB selected top 20,dGeXdSMah56gEHwZNaRQKA,Juan's Flying Burrito,2021-12,1,1,0.200017,0.2,3.0,...,0.833333,3.0,0.666667,0.000000,3.0,2.779347,0.000022,0.666667,2.0,Model correctly flagged a future pulse; inspec...
4,missed_pulse,ML: HGB selected top 20,CcAtO2dgxzG58x6sze8dlg,Loretta's Authentic Pralines,2021-08,1,0,0.043566,0.2,8.0,...,5.333333,18.0,0.722222,0.277778,18.0,1.434220,0.000040,0.722222,2.0,A pulse occurred despite a low model decision;...


In [7]:
print("Social graph summary")
for key, value in graph_summary.items():
    print(f"{key}: {value}")

print("\nForecasting dataset summary")
for key, value in feature_summary.items():
    print(f"{key}: {value}")

Social graph summary
active_review_threshold: 5
threshold_candidates: [2, 3, 5, 10, 20]
edge_weight_formula: 1 + log1p(shared_business_count) + category_jaccard
reviewing_users: 245421
matched_user_profiles: 245419
active_users: 26598
graph_nodes: 26598
graph_edges: 116558
mean_edge_weight: 1.833072733525831
mean_edge_shared_business_count: 2.048550936014688
mean_edge_category_jaccard: 0.2671618532172656
connected_components: 11508
largest_component_size: 14965
isolated_active_users: 11387
community_method: weighted_louvain_largest_component
communities_assigned: 63
threshold_sensitivity_output: C:\Users\mehdi\OneDrive\Documents\community-forecasting-yelp\data\processed\new_orleans\active_reviewer_threshold_sensitivity.csv
output: C:\Users\mehdi\OneDrive\Documents\community-forecasting-yelp\data\processed\new_orleans\user_network_features.csv

Forecasting dataset summary
min_total_reviews: 100
min_active_months: 36
business_count: 876
row_count: 68249
feature_month_min: 2015-01
feature

In [8]:
for _, row in regression_summary.iterrows():
    split = row["split"]
    best_vs_last_month = row["best_vs_last_month_relative_change"] * 100
    all_vs_hist = row["all_vs_historical_relative_change"] * 100
    all_vs_business = row["all_vs_business_relative_change"] * 100
    print(f"{split} regression:")
    print(f"  Best model: {row['best_model']} ({row['best_family']}) with WAPE={row['best_WAPE']:.4f}")
    print(f"  Best vs last-month baseline: {best_vs_last_month:+.2f}%")
    print(f"  HGB all modalities WAPE: {row['hgb_all_WAPE']:.4f}")
    print(f"  Poisson all modalities WAPE: {row['poisson_all_WAPE']:.4f}")
    print(f"  Selected HGB WAPE: {row['selected_hgb_WAPE']:.4f}")
    print(f"  All modalities vs historical: {all_vs_hist:+.2f}%")
    print(f"  All modalities vs historical+business: {all_vs_business:+.2f}%")

print()
for _, row in pulse_summary.iterrows():
    split = row["split"]
    print(f"{split} attention pulses:")
    print(f"  Positive rate: {row['positive_rate']:.3f}")
    print(f"  Best thresholded model: {row['best_model']} ({row['best_family']}) with F1={row['best_F1']:.4f}, PR-AUC={row['best_PR_AUC']:.4f}, Brier={row['best_Brier']:.4f}")
    print(f"  Best model threshold: {row['best_threshold']:.3f}; validation F1={row['best_validation_F1']:.4f}")
    print(f"  Best Brier model: {row['best_brier_model']} with Brier={row['best_brier']:.4f}")
    print(f"  HGB selected top 20: F1={row['selected_hgb_F1']:.4f}, PR-AUC={row['selected_hgb_PR_AUC']:.4f}, Brier={row['selected_hgb_Brier']:.4f}")
    print(f"  Logistic all modalities: F1={row['logistic_all_F1']:.4f}, PR-AUC={row['logistic_all_PR_AUC']:.4f}")
    print(f"  Best precision@10%: {row['best_precision_at_10pct_model']} with precision={row['best_precision_at_10pct']:.4f}, recall={row['best_recall_at_10pct']:.4f}")

primary_covid_test regression:
  Best model: Baseline: last month (temporal_baseline) with WAPE=0.6643
  Best vs last-month baseline: +0.00%
  HGB all modalities WAPE: 0.9352
  Poisson all modalities WAPE: 0.8712
  Selected HGB WAPE: 0.9048
  All modalities vs historical: +2.43%
  All modalities vs historical+business: +2.77%
secondary_pre_covid_test regression:
  Best model: ML: HGB all modalities (HistGradientBoostingRegressor) with WAPE=0.3806
  Best vs last-month baseline: -14.91%
  HGB all modalities WAPE: 0.3806
  Poisson all modalities WAPE: 0.4495
  Selected HGB WAPE: 0.3920
  All modalities vs historical: -1.00%
  All modalities vs historical+business: +0.81%

primary_covid_test attention pulses:
  Positive rate: 0.102
  Best thresholded model: Baseline: rising recent activity (rule_baseline) with F1=0.2809, PR-AUC=0.1481, Brier=0.2002
  Best model threshold: 0.500; validation F1=0.2156
  Best Brier model: ML: HGB selected top 20 with Brier=0.0920
  HGB selected top 20: F1=0.2

## Interpretation Structure

1. **COVID-era disruption:** raw review-count forecasting becomes harder in the 2020-2021 test window. The last-month baseline remains difficult to beat because it adapts quickly to collapsed or irregular activity.
2. **Attention-pulse framing:** attention pulses better match the project objective because they ask whether a business is about to receive unusual community attention, not merely whether already-popular businesses will keep receiving reviews.
3. **SNA and NLP:** social exposure, recent language, and capped TF-IDF topic indicators add interpretability and modality coverage, but they do not automatically create large predictive lift after temporal and business signals are included.
4. **Pulse precursors:** pulse rows can be examined through review momentum, recent text volume, reviewer centrality, and topical language to explain what kinds of signals precede unusual attention.
5. **Multimodal analytics:** combining modalities is analytically valuable because it exposes preparation, leakage, and interpretation challenges. Predictively, however, more modalities are not automatically better.
6. **Decision framing:** pulse prediction should be interpreted as rare-event ranking and risk triage. Top-k metrics and case studies are useful because analysts may only inspect the highest-risk business-months.

## Final Position

This project should be presented as an interpretable multimodal analytics study of **local community attention dynamics**, not as a claim that the most complex model always wins.

The strongest academic story is:

- **COVID-era disruption makes raw count forecasting harder.** Simple temporal baselines remain powerful when the environment changes sharply.
- **Attention pulses are a better framing for community attention.** They focus on unusual short-term shifts relative to each business's own recent baseline.
- **SNA and NLP add interpretation more than large standalone lift.** Social exposure, recent language, and TF-IDF topic indicators help explain possible precursors, even when their predictive gains are modest.
- **Case studies make the model auditable.** Correct predictions, missed pulses, and false alarms show where digital traces anticipate attention and where Yelp data remains incomplete.
- **Multimodal analytics is valuable, but not automatically better.** The project shows the practical challenges of preparing, aligning, modeling, and interpreting multiple data modalities responsibly.

Key limitations:

- Yelp friendship links are static.
- SNA features measure exposure, not causal influence.
- NLP features are lightweight lexicon/text-length/TF-IDF signals.
- COVID-era disruption changes predictability.
- Review activity is a proxy for Yelp attention, not revenue or true customer volume.
- Static business metadata may include end-of-dataset information.
- Calibration is diagnostic, not a guarantee that probabilities transfer outside this dataset.